# Fidelity App - Churn Analysis

This notebook analyses user churn for the **Fidely App** loyalty platform.

**Churn definition:** A user is considered *churned* if they have not visited any business in the last 30 days relative to the most recent activity date in the dataset.

**Data source:** `PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE` (star schema):
- `FACT_VISITS` — every check-in, redemption, referral, and VIP activation
- `DIM_USERS` — consumer profiles (status, registration date, preferred zone)
- `DIM_BUSINESSES` — merchant profiles (category, loyalty type, neighbourhood)
- `DIM_LOYALTY_TYPE` — loyalty program reference
- `DIM_TIME` — date dimension

## 1. Compute Churn Flags per User

We label each user as **churned** if their last visit is more than 30 days before the dataset's maximum date.

In [2]:
%load_ext sql


ModuleNotFoundError: No module named 'sql'

In [1]:

%%sql -r churn_flags

WITH dataset_boundary AS (
    SELECT MAX(VISIT_TIMESTAMP)::DATE AS max_date
    FROM PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.FACT_VISITS
),
user_activity AS (
    SELECT
        u.USER_KEY,
        u.USER_ID,
        u.FIRST_NAME || ' ' || u.LAST_NAME AS FULL_NAME,
        u.STATUS,
        u.REGISTRATION_DATE,
        u.PREFERRED_ZONE,
        MIN(fv.VISIT_TIMESTAMP)::DATE  AS FIRST_VISIT,
        MAX(fv.VISIT_TIMESTAMP)::DATE  AS LAST_VISIT,
        COUNT(*)                       AS TOTAL_VISITS,
        COUNT(DISTINCT fv.BUSINESS_KEY) AS DISTINCT_BUSINESSES,
        SUM(fv.POINTS_EARNED)          AS TOTAL_POINTS,
        SUM(fv.CASHBACK_AMOUNT)        AS TOTAL_CASHBACK
    FROM PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.DIM_USERS u
    LEFT JOIN PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.FACT_VISITS fv
        ON u.USER_KEY = fv.USER_KEY
    GROUP BY u.USER_KEY, u.USER_ID, FULL_NAME, u.STATUS, u.REGISTRATION_DATE, u.PREFERRED_ZONE
)
SELECT
    ua.*,
    DATEDIFF('day', ua.LAST_VISIT, db.max_date) AS DAYS_SINCE_LAST_VISIT,
    CASE
        WHEN ua.LAST_VISIT IS NULL THEN 'never_visited'
        WHEN DATEDIFF('day', ua.LAST_VISIT, db.max_date) > 30 THEN 'churned'
        ELSE 'active'
    END AS CHURN_STATUS
FROM user_activity ua
CROSS JOIN dataset_boundary db
ORDER BY DAYS_SINCE_LAST_VISIT DESC NULLS FIRST

UsageError: Cell magic `%%sql` not found.


## 2. Overall Churn Rate

In [ ]:
import pandas as pd

summary = churn_flags.groupby('CHURN_STATUS').agg(
    user_count=('USER_KEY', 'count'),
    avg_total_visits=('TOTAL_VISITS', 'mean'),
    avg_points=('TOTAL_POINTS', 'mean'),
    avg_cashback=('TOTAL_CASHBACK', 'mean')
).reset_index()

total_users = summary['user_count'].sum()
summary['pct_of_total'] = (summary['user_count'] / total_users * 100).round(1)

print(f"Total users: {total_users}")
print()
print(summary.to_string(index=False))

NameError: name 'churn_flags' is not defined

## 3. Churn Rate Visualisation

In [ ]:
import matplotlib.pyplot as plt

colors = {'active': '#2ecc71', 'churned': '#e74c3c', 'never_visited': '#95a5a6'}
labels = summary['CHURN_STATUS']
sizes = summary['user_count']
chart_colors = [colors.get(s, '#bdc3c7') for s in labels]

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=chart_colors,
    autopct='%1.1f%%', startangle=140, textprops={'fontsize': 12}
)
ax.set_title('User Churn Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## 4. Churn by Loyalty Type

Do certain loyalty programs retain users better than others?

In [ ]:
%%sql -r churn_by_loyalty
WITH dataset_boundary AS (
    SELECT MAX(VISIT_TIMESTAMP)::DATE AS max_date
    FROM PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.FACT_VISITS
),
user_loyalty AS (
    SELECT
        fv.USER_KEY,
        lt.LOYALTY_TYPE_NAME,
        MAX(fv.VISIT_TIMESTAMP)::DATE AS LAST_VISIT
    FROM PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.FACT_VISITS fv
    JOIN PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.DIM_LOYALTY_TYPE lt
        ON fv.LOYALTY_TYPE_KEY = lt.LOYALTY_TYPE_KEY
    GROUP BY fv.USER_KEY, lt.LOYALTY_TYPE_NAME
)
SELECT
    ul.LOYALTY_TYPE_NAME,
    COUNT(*)                                                          AS TOTAL_USERS,
    SUM(CASE WHEN DATEDIFF('day', ul.LAST_VISIT, db.max_date) > 30 THEN 1 ELSE 0 END) AS CHURNED_USERS,
    ROUND(CHURNED_USERS / TOTAL_USERS * 100, 1)                      AS CHURN_RATE_PCT
FROM user_loyalty ul
CROSS JOIN dataset_boundary db
GROUP BY ul.LOYALTY_TYPE_NAME
ORDER BY CHURN_RATE_PCT DESC

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(
    churn_by_loyalty['LOYALTY_TYPE_NAME'],
    churn_by_loyalty['CHURN_RATE_PCT'],
    color='#e74c3c', edgecolor='white'
)
ax.set_xlabel('Churn Rate (%)')
ax.set_title('Churn Rate by Loyalty Program Type', fontsize=13, fontweight='bold')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Churn by Business Category

Which merchant categories see the most user churn?

In [ ]:
%%sql -r churn_by_category
WITH dataset_boundary AS (
    SELECT MAX(VISIT_TIMESTAMP)::DATE AS max_date
    FROM PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.FACT_VISITS
),
user_category AS (
    SELECT
        fv.USER_KEY,
        b.CATEGORY,
        MAX(fv.VISIT_TIMESTAMP)::DATE AS LAST_VISIT
    FROM PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.FACT_VISITS fv
    JOIN PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.DIM_BUSINESSES b
        ON fv.BUSINESS_KEY = b.BUSINESS_KEY
    GROUP BY fv.USER_KEY, b.CATEGORY
)
SELECT
    uc.CATEGORY,
    COUNT(*)                                                          AS TOTAL_USERS,
    SUM(CASE WHEN DATEDIFF('day', uc.LAST_VISIT, db.max_date) > 30 THEN 1 ELSE 0 END) AS CHURNED_USERS,
    ROUND(CHURNED_USERS / TOTAL_USERS * 100, 1)                      AS CHURN_RATE_PCT
FROM user_category uc
CROSS JOIN dataset_boundary db
GROUP BY uc.CATEGORY
ORDER BY CHURN_RATE_PCT DESC

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    churn_by_category['CATEGORY'],
    churn_by_category['CHURN_RATE_PCT'],
    color='#3498db', edgecolor='white'
)
ax.set_xlabel('Churn Rate (%)')
ax.set_title('Churn Rate by Business Category', fontsize=13, fontweight='bold')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Monthly Churn Trend

How does the monthly churn rate evolve over time?

In [ ]:
%%sql -r monthly_churn
WITH monthly_users AS (
    SELECT
        DATE_TRUNC('month', VISIT_TIMESTAMP)::DATE AS MONTH,
        COUNT(DISTINCT USER_KEY)                   AS ACTIVE_USERS
    FROM PINNACLE_FINANCIAL_DEMO_33.ANALYTICS_ZONE.FACT_VISITS
    GROUP BY MONTH
),
month_pairs AS (
    SELECT
        curr.MONTH,
        prev.ACTIVE_USERS                                  AS PREV_ACTIVE,
        curr.ACTIVE_USERS                                  AS CURR_ACTIVE,
        prev.ACTIVE_USERS - curr.ACTIVE_USERS              AS LOST_USERS
    FROM monthly_users curr
    JOIN monthly_users prev
        ON curr.MONTH = DATEADD('month', 1, prev.MONTH)
)
SELECT
    MONTH,
    PREV_ACTIVE,
    CURR_ACTIVE,
    GREATEST(LOST_USERS, 0)                                          AS LOST_USERS,
    ROUND(GREATEST(LOST_USERS, 0) / NULLIF(PREV_ACTIVE, 0) * 100, 1) AS MONTHLY_CHURN_RATE_PCT
FROM month_pairs
ORDER BY MONTH

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(monthly_churn['MONTH'].astype(str), monthly_churn['LOST_USERS'],
        color='#e74c3c', alpha=0.6, label='Lost Users')
ax1.set_xlabel('Month')
ax1.set_ylabel('Lost Users', color='#e74c3c')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly_churn['MONTH'].astype(str), monthly_churn['MONTHLY_CHURN_RATE_PCT'],
         color='#2c3e50', marker='o', linewidth=2, label='Churn Rate %')
ax2.set_ylabel('Churn Rate (%)', color='#2c3e50')

ax1.set_title('Monthly Churn Trend', fontsize=13, fontweight='bold')
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.show()

## 7. Days-Since-Last-Visit Distribution (Churned vs Active)

A histogram showing the distribution of inactivity days, split by churn status.

In [ ]:
import matplotlib.pyplot as plt

visited = churn_flags[churn_flags['CHURN_STATUS'] != 'never_visited'].copy()

fig, ax = plt.subplots(figsize=(10, 5))
for status, color in [('active', '#2ecc71'), ('churned', '#e74c3c')]:
    subset = visited[visited['CHURN_STATUS'] == status]
    ax.hist(subset['DAYS_SINCE_LAST_VISIT'], bins=30, alpha=0.6,
            color=color, label=status, edgecolor='white')

ax.axvline(x=30, color='black', linestyle='--', linewidth=1, label='30-day threshold')
ax.set_xlabel('Days Since Last Visit')
ax.set_ylabel('Number of Users')
ax.set_title('Inactivity Distribution by Churn Status', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

Key findings from this churn analysis:
- **Overall churn rate** — proportion of users inactive for 30+ days
- **Loyalty type impact** — which reward mechanisms retain users best
- **Category patterns** — business categories with highest attrition
- **Trend over time** — whether churn is improving or worsening month-over-month

Next steps could include building a predictive churn model, segmenting by preferred zone / neighbourhood, or designing targeted re-engagement campaigns.